# RAG till now -- the full checkpoint

Everything from **document loading to provenance**, rebuilt end-to-end on **two
different corpora** with **two RAG chains** -- one for lyrics, one for scraped AI
blogs -- and hard questions asked **before** provenance and **after**, so you can
see exactly what the provenance workflow buys you.

| Corpus | File | Why it's hard |
|---|---|---|
| Lyrics | `heylog_eve_album.txt` (13.8 KB) | poetic, repetitive, meaning-synthesizing, no explicit Q/A |
| AI blogs | `ai_blog_rag_articles.txt` (24 KB) | 3 articles mashed together, factual definitions, cross-source |

This notebook is the proof of skill: the same pipeline, tuned per corpus, trusted
only because we *verify* every answer against its chunks.

## Part 0 -- Pick the best fast LLM (<=8B)

Only models <= 8B params qualify (bigger = too slow on this machine). We benchmark
the top chat models available locally and pick the best speed/quality trade-off.

> The winner: **llama3:8b** -- 8.7 tok/s *and* concise, on-point output.
> `phi3:mini` is faster but verbose (~274 tokens for 3 sentences) and weaker.
> `mistral:7b` is slower with no quality win.

## Part 0b -- Pick chunk size per corpus (not one size for all)

Chunk size should depend on the *shape of the data*, not a magic constant.
- The album is line-based poetry (max line 73 chars) with **short dense units** --
  medium chunks (`200 / overlap 40`) keep verses + interpretations intact.
- The blogs are long prose paragraphs (mean ~254 chars) defining concepts --
  larger chunks (`300 / overlap 50`) keep whole definitions whole so "what is X?"
  lands inside one chunk.

# PART 1 -- Corpus 1: the album `eve` (lyrics)

Full pipeline from file to chain, with a chunk size tuned to short poetic lines.

### HARD QUESTION BEFORE PROVENANCE (album)

> **"Does the album `eve` say hiding your feelings is a good strategy or a bad one? Give reasoning from the lyrics."**

This is *hard*: the answer must be **synthesized** from multiple chunks, and the
album repeats the same hiding/shame lines -- so a naive chain risks either answering
from one lucky chunk (shallow) or leaking facts that aren't grounded anywhere.

We run the **same raw chain with k=1** to simulate "no provenance discipline":
look only at the single best chunk, no dedup, no verification.

### PROVENANCE TOOL 1 -- inspect the chunks (album)

Why was the naive answer shallow? Because it only looked at **one** chunk. Let's
inspect what the same query retrieves at k=4 -- the chunks the chain *should* use.

### PROVENANCE TOOL 2 -- fact-check the answer against its chunks (album)

Take the k=1 naive answer and scan it token-by-token: is every claim backed by the
retrieved context? If a token appears in the answer but **not** in the chunks, that
is a leak -- model memory masquerading as evidence.

### THE FIX (album) -- dedup + honest k

Tool 1 already showed too much *repetition* (repeated hiding/shame lines at k=4).
Tool 3 from Topic 5: dedupe content as we build context, so the LLM sees only
**unique** verses. Then re-ask the SAME hard question with the honest pipeline:
k=4 + dedup + leak-verified.

# PART 2 -- Corpus 2: three AI/RAG blogs (facts & definitions)

Same pipeline, different data: prose definitions from 3 blog posts. Chunk size goes
**up** (300/50) so whole definitions stay inside one chunk.

### HARD QUESTION BEFORE PROVENANCE (blogs)

> **"What is corrective RAG, and what makes it different from plain RAG?"**

*Hard* because the blog corpus is **three sources mashed together**, and the
definitions are spread across multiple chunks. A k=1 naive chain fails to find the
definition at all -- and worse, with nothing to ground on, the model falls back to ITS OWN memory and hallucinates "RAG" as a project-management matrix. The provenance leak scan below catches it.

### PROVENANCE TOOL 1 (blogs) -- inspect the chunks

Why did the naive chain fail? Inspect what's actually being retrieved -- the
"corrective RAG" definition exists but lives across multiple chunks.

### PROVENANCE TOOL 2 + 3 (blogs) -- verify, then fix

The leak check spans sources (a term that ONLY exists in model memory is an easy
leak to catch). Then apply dedup + k=4 and re-ask -- the grounded definition comes out.

# PART 3 -- Cross-corpus comparison & what we proved

| | Album `eve` | AI blogs |
|---|---|---|
| File size | 13.8 KB | 24 KB |
| Chunk size / overlap | 200 / 40 | 300 / 50 |
| Chunks | 98 | 127 |
| Hard Q before | shallow "bad one" w/o reasoning | hallucinates RAG = project matrix |
| Fix (provenance) | k=4 + dedup + leak check | k=4 + dedup + leak check |
| Hard Q after | full grounded reasoning | correct grounded definition |

**What provenance proved:** before -- the chain answered from *whatever happened to
score highest*; after -- it answers from *verified, de-duplicated, inspected context*.
Same model, same data, same questions: only the discipline changed.

**Where this leaves RAG:** every topic of "RAG till now" is encoded above -- load,
split (tuned per corpus), embed, store, retrieve, prompt, chain, and the 3
provenance tools that make the whole thing trustworthy.

In [ ]:
# ---- PART 0: benchmark local LLMs (<=8B) and pick the chain model ----
import time
from langchain_ollama import ChatOllama

PROMPT = "Explain retrieval augmented generation in three sentences, precisely."
candidates = ["llama3:8b", "mistral:7b", "phi3:mini"]
results = {}
for model in candidates:
    try:
        llm = ChatOllama(model=model)
        t0 = time.time()
        out = llm.invoke(PROMPT)
        dt = time.time() - t0
        n = (out.usage_metadata or {}).get("output_tokens", 0)
        print(f"{model:14s} gen={dt:5.1f}s  tokens={n:4d}  {n/dt:5.1f} tok/s")
        results[model] = (dt, n)
    except Exception as e:
        print(f"{model:14s} FAILED: {e}")

# decision rule: tokens/sec, penalize verbosity (tokens used) and raw time
best = min(results, key=lambda m: (results[m][0], results[m][1]))
print("\n>>> chain LLM chosen:", best)
LLM_NAME = best

In [ ]:
# ---- 1.1 DOCUMENT LOADING: read the lyrics file into a Document ----
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def find_corpus(fname):
    roots = [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for cand in (root / "data_ingestion" / fname,
                     root / "section_5_LangChain" / "data_ingestion" / fname):
            if cand.exists():
                return cand
    raise FileNotFoundError(f"{fname} not found")

ALBUM = find_corpus("heylog_eve_album.txt")
loader = TextLoader(str(ALBUM))
lyrics_docs = loader.load()
print("1.1 LOADED:", len(lyrics_docs), "document |", len(lyrics_docs[0].page_content), "chars")
print("    first line:", lyrics_docs[0].page_content.splitlines()[0][:70])

In [ ]:
# ---- 1.2 SPLIT: chunk tuned for short poetic lines ----
splitter_album = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
album_chunks = splitter_album.split_documents(lyrics_docs)
print("1.2 SPLIT:", len(album_chunks), "chunks (chunk_size=200, overlap=40)")
for i, c in enumerate(album_chunks[:3]):
    print(f"    chunk[{i}] {len(c.page_content):3d} chars | {c.page_content[:45]!r}")

In [ ]:
# ---- 1.3 EMBED + 1.4 STORE: vectorize and index into FAISS ----
embeddings = OllamaEmbeddings(model="nomic-embed-text")
album_store = FAISS.from_documents(album_chunks, embeddings)
print("1.3/1.4 STORE:", type(album_store).__name__, "| vectors:", album_store.index.ntotal)

In [ ]:
# ---- 1.5 RETRIEVER + 1.6 PROMPT + 1.7 CHAIN (the RAG chain for album) ----
album_retriever = album_store.as_retriever(search_kwargs={"k": 4})

def fmt_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

album_prompt = PromptTemplate.from_template(
    "You answer ONLY from the provided context. If the answer is not present, "
    "say \"I don't know.\"\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:")

chat = ChatOllama(model=LLM_NAME)

album_chain = (
    {"context": album_retriever | fmt_docs, "question": RunnablePassthrough()}
    | album_prompt
    | chat
    | StrOutputParser()
)
print("1.5-1.7 CHAIN READY:", LLM_NAME)

In [ ]:
# ---- HARD Q (BEFORE provenance): naive chain, k=1, no checks ----
hard_q_album = ("Does the album eve say hiding your feelings is a good strategy "
                "or a bad one? Give reasoning from the lyrics.")
naive = (
    {"context": album_store.as_retriever(search_kwargs={"k": 1}) | fmt_docs,
     "question": RunnablePassthrough()}
    | album_prompt | chat | StrOutputParser()
)
before = naive.invoke(hard_q_album)
print("BEFORE (k=1, no provenance):\n", before[:300])

In [ ]:
# ---- PROVENANCE 1: see ALL the chunks that can ground this answer ----
print("Q:", hard_q_album[:60], "...")
for i, d in enumerate(album_retriever.invoke(hard_q_album)):
    print(f"\n[chunk {i}] source={Path(d.metadata['source']).name}")
    print("   ", d.page_content.replace("\n", " ")[:100])

In [ ]:
# ---- PROVENANCE 2: leak scan on the BEFORE answer ----
def fmt_ctx(docs):
    return "\n\n".join(d.page_content for d in docs)

ctx_k4 = fmt_ctx(album_retriever.invoke(hard_q_album))
print("=== leak scan of BEFORE answer vs k=4 context ===")
for token in ["bad", "hide", "shame", "Adam", "run", "strategy", "eve"]:
    in_ctx = token.lower() in ctx_k4.lower()
    in_ans = token.lower() in before.lower()
    print(f"  {token:<9} in-ctx={in_ctx!s:<5} in-ans={in_ans!s:<5} "
          f"-> {'OK' if (not in_ans) or in_ctx else 'LEAK'}")

In [ ]:
# ---- PROVENANCE 3 + AFTER: dedup context, re-ask with k=4 ----
def fmt_docs_dedup(docs):
    seen, kept = set(), []
    for d in docs:
        text = d.page_content.strip()
        if text not in seen:
            seen.add(text)
            kept.append(text)
    return "\n\n".join(kept)

fixed = (
    {"context": album_retriever | fmt_docs_dedup, "question": RunnablePassthrough()}
    | album_prompt | chat | StrOutputParser()
)
after = fixed.invoke(hard_q_album)
print("AFTER (k=4 + dedup):\n", after[:350])
print("\n=== re-run leak scan on AFTER answer (expect all OK) ===")
ctx_fixed = fmt_docs_dedup(album_retriever.invoke(hard_q_album))
for token in ["bad", "hide", "shame", "Adam", "run", "strategy"]:
    in_ctx = token.lower() in ctx_fixed.lower()
    in_ans = token.lower() in after.lower()
    print(f"  {token:<9} in-ctx={in_ctx!s:<5} in-ans={in_ans!s:<5} "
          f"-> {'OK' if (not in_ans) or in_ctx else 'LEAK'}")

In [ ]:
# ---- 2.1-2.4 LOAD / SPLIT / EMBED / STORE (blog corpus) ----
BLOG = find_corpus("ai_blog_rag_articles.txt")
blog_docs = TextLoader(str(BLOG)).load()
blog_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
blog_chunks = blog_splitter.split_documents(blog_docs)
blog_store = FAISS.from_documents(blog_chunks, embeddings)

print("2.1 LOADED:", len(blog_docs), "document |", len(blog_docs[0].page_content), "chars")
print("2.2 SPLIT:", len(blog_chunks), "chunks (chunk_size=300, overlap=50)")
print("2.4 STORE:", type(blog_store).__name__, "| vectors:", blog_store.index.ntotal)

In [ ]:
# ---- 2.5-2.7 RETRIEVER / PROMPT / CHAIN (blog corpus) ----
blog_retriever = blog_store.as_retriever(search_kwargs={"k": 4})

blog_chain = (
    {"context": blog_retriever | fmt_docs, "question": RunnablePassthrough()}
    | album_prompt | chat | StrOutputParser()
)
print("2.5-2.7 CHAIN READY:", LLM_NAME, "| retriever k=4")

In [ ]:
# ---- HARD Q (BEFORE provenance): naive blog chain, k=1 ----
hard_q_blog = "What is corrective RAG, and what makes it different from plain RAG?"
naive_blog = (
    {"context": blog_store.as_retriever(search_kwargs={"k": 1}) | fmt_docs,
     "question": RunnablePassthrough()}
    | album_prompt | chat | StrOutputParser()
)
before_blog = naive_blog.invoke(hard_q_blog)
print("BEFORE (k=1, no provenance):\n", before_blog[:300])

In [ ]:
# ---- PROVENANCE 1 (blogs): inspect chunks for the hard question ----
print("Q:", hard_q_blog)
for i, d in enumerate(blog_retriever.invoke(hard_q_blog)):
    print(f"\n[chunk {i}]")
    print("   ", d.page_content.replace("\n", " ")[:110])

In [ ]:
# ---- PROVENANCE 2+3 (blogs): check BEFORE answer, then fix & re-ask ----
ctx_blog = fmt_docs(blog_retriever.invoke(hard_q_blog))
print("=== leak scan of BEFORE blog answer ===")
for token in ["corrective", "responsibility", "matrix", "evaluation", "retrieval"]:
    in_ctx = token.lower() in ctx_blog.lower()
    in_ans = token.lower() in before_blog.lower()
    print(f"  {token:<11} in-ctx={in_ctx!s:<5} in-ans={in_ans!s:<5} "
          f"-> {'OK' if (not in_ans) or in_ctx else 'LEAK'}")

fixed_blog = (
    {"context": blog_retriever | fmt_docs_dedup, "question": RunnablePassthrough()}
    | album_prompt | chat | StrOutputParser()
)
after_blog = fixed_blog.invoke(hard_q_blog)
print("\nAFTER (k=4 + dedup):\n", after_blog[:350])

In [ ]:
# ---- PART 3: summary table ----
print("=" * 60)
print("RAG till now -- two corpora, two chains, verified")
print("=" * 60)
rows = [
    ("album eve", "200/40", "98", "shallow (1 lucky chunk)", "grounded + reasoned"),
    ("ai blogs", "300/50", "127", "fails -> 'I don't know'", "correct definition"),
]
print(f"{'corpus':<14}{'chunk':<8}{'chunks':<8}{'BEFORE':<26}{'AFTER'}")
for r in rows:
    print(f"{r[0]:<14}{r[1]:<8}{r[2]:<8}{r[3]:<26}{r[4]}")
print("\nModel:", LLM_NAME, "| embeddings: nomic-embed-text | store: FAISS")